# 03 - Initial profile from the raw Apple Watch export


## Questions

- How do load, rest ratio, heart-rate intensity, and pace relate?
- Which strokes dominate total distance and time?
- Which correlations are strong enough to deserve regression/inference work?
- Which fields should become prepared data sources for later notebooks?

Key formulas used by the parser:

$$
\text{pace}_{50m} = \frac{\text{swim time seconds}}{\text{distance meters}} \times 50
$$

Code-style implementation:

```python
pace_50_s = swim_time / distance * 50
```

$$
\text{rest ratio} = \frac{\text{rest time seconds}}{\text{total duration seconds}}
$$

Code-style implementation:

```python
rest_ratio = rest_time / duration
```

$$
\text{simple HR load} = \frac{\text{distance meters} \times \text{average HR}}{1000}
$$

Code-style implementation:

```python
simple_hr_load = distance * avg_hr / 1000
```

**Expected output:** The cell should run without errors and make the required packages, paths, or helper tools available for the rest of the notebook.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from go_swim_analysis.analysis import parse_export

raw_export = ROOT / "data/raw/go-swim-export.json"
parsed = parse_export(raw_export)
parsed.summary


{'workout_count': 483,
 'split_count': 29818,
 'date_min': '2024-05-31T23:13:14+00:00',
 'date_max': '2026-06-17T14:08:13+00:00',
 'total_distance_km': 1500.7811497588355,
 'total_swim_hours': 443.90434647887946,
 'total_rest_hours': 519.4203100544214,
 'metrics': {'distance_m': {'n': 483,
   'mean': 3107.207349397175,
   'median': 3223.2600860595703,
   'min': 0,
   'max': 9144.000244140625,
   'p25': 2514.600067138672,
   'p75': 3874.77010345459},
  'duration_s': {'n': 483,
   'mean': 7180.059551801,
   'median': 7523.739886045456,
   'min': 0.06355404853820801,
   'max': 19315.55944800377,
   'p25': 6589.188832044601,
   'p75': 8286.428385019302},
  'swim_time_s': {'n': 483,
   'mean': 3308.6038246873004,
   'median': 3333.0737649202347,
   'min': 0.06355404853820801,
   'max': 12269.025874614716,
   'p25': 2701.1226397156715,
   'p75': 3867.282904982567},
  'rest_time_s': {'n': 483,
   'mean': 3871.4557271137,
   'median': 4015.82678771019,
   'min': 0,
   'max': 10884.02497124672,

### Code cell note

**Expected output:** Expect output that helps confirm this step worked and gives evidence for the next interpretation in the analysis.


In [2]:
parsed.stroke_rows

[{'stroke': 'Freestyle',
  'distance_m': 978763.0338247075,
  'distance_km': 978.7630338247075,
  'duration_s': 924707.930352211,
  'duration_h': 256.8633139867253,
  'strokes': 366913.5349302177,
  'pace_50_s': 47.238601091151466},
 {'stroke': 'Kickboard',
  'distance_m': 186040.0831279214,
  'distance_km': 186.04008312792138,
  'duration_s': 276030.25930178165,
  'duration_h': 76.67507202827268,
  'strokes': 3739.2518757405123,
  'pace_50_s': 74.18569553959587},
 {'stroke': 'Backstroke',
  'distance_m': 125505.01490771533,
  'distance_km': 125.50501490771533,
  'duration_s': 143664.25009548664,
  'duration_h': 39.90673613763518,
  'strokes': 37883.09713808638,
  'pace_50_s': 57.23446596979568},
 {'stroke': 'Breaststroke',
  'distance_m': 116322.7400087067,
  'distance_km': 116.3227400087067,
  'duration_s': 127305.51902759075,
  'duration_h': 35.36264417433076,
  'strokes': 50876.06817881595,
  'pace_50_s': 54.72082200697043},
 {'stroke': 'Butterfly',
  'distance_m': 70967.7071549859

### Code cell note

**Expected output:** Expect relationship measures between variables. Interpret direction and strength, but remember correlation does not prove causation.


In [3]:
parsed.correlations

{'workout_distance_vs_avg_hr': 0.4852274150754616,
 'workout_distance_vs_pace_50': -0.4113273888620279,
 'workout_distance_vs_rest_ratio': 0.06978989860276685,
 'rest_ratio_vs_pace_50': -0.5663064501565555,
 'avg_hr_vs_pace_50': -0.454167770341227,
 'hr90_vs_pace_50': -0.10739678460378205,
 'simple_hr_load_vs_pace_50': -0.4192333183791719,
 'split_pace_vs_workout_avg_hr': -0.11769934677961108,
 'split_pace_vs_workout_rest_ratio': 0.06306434643565847,
 'split_pace_vs_workout_distance': -0.15409602896805863}

## Initial interpretation

The first-pass report is generated at `reports/initial_profile.md`. Use that report to decide which relationships deserve deeper AP Statistics inference and which time trends deserve AP Precalculus modeling.

Correlation is the main first-pass relationship metric:

$$
r =
\frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{(n - 1)s_xs_y}
$$

Code-style implementation:

```python
correlation(column(rows, "distance_m"), column(rows, "avg_hr"))
```

Values near `1` or `-1` suggest a stronger linear relationship. Values near `0` suggest little linear relationship.
